In [ ]:
# | default_exp insights.trends

# Insights Trends
> Detect rising and declining queries by comparing consecutive periods.

In [ ]:
#| export
from datetime import datetime, timedelta
from sqlmodel import Session
from seo_rat.gsc.queries import get_top_queries


In [ ]:
# | export
def apply_strftime(date: datetime):
    return date.strftime("%Y-%m-%d")
def get_date_ranges_for_comparison(days: int = 30
                                   ) -> tuple[tuple[str, str], tuple[str, str]]:
    "Return two consecutive date ranges for period comparison, accounting for 3-day GSC delay."
    fmt = lambda d: d.strftime("%Y-%m-%d")
    latest = datetime.now() - timedelta(days=3)
    period_a_end = latest
    period_a_start = latest - timedelta(days=days)
    period_b_end = period_a_start
    period_b_start = period_b_end - timedelta(days=days)
    return (fmt(period_a_start), fmt(period_a_end)), (fmt(period_b_start), fmt(period_b_end))

In [ ]:
# | export
def compare_periods(recent: list[dict],
                    previous: list[dict]
                    ) -> list[dict]:
    "Compare two periods of query data and return trend analysis sorted by impression change."
    prev_by_query = {r["query"]: r for r in previous}
    trends = []
    for r in recent:
        p = prev_by_query.get(r["query"], {"total_clicks": 0, "total_impressions": 0, "avg_position": 100})
        click_change = r["total_clicks"] - p["total_clicks"]
        impression_change = r["total_impressions"] - p["total_impressions"]
        position_change = p["avg_position"] - r["avg_position"]
        trends.append({"query": r["query"],
                       "recent_clicks": r["total_clicks"], "prev_clicks": p["total_clicks"],
                       "click_change": click_change,
                       "recent_impressions": r["total_impressions"], "prev_impressions": p["total_impressions"],
                       "impression_change": impression_change,
                       "recent_position": r["avg_position"], "prev_position": p["avg_position"],
                       "position_change": position_change,
                       "trend": "rising" if impression_change > 0 and position_change > 0
                       else "declining" if impression_change < 0 or position_change < 0
                       else "stable"})
    return sorted(trends, key=lambda t: t["impression_change"], reverse=True)


In [ ]:
# | export
def detect_query_trends(session: Session,
                        site_url: str,
                        page_path: str | None = None,
                        days: int = 30,
                        limit: int = 100
                        ) -> list[dict]:
    "Detect rising and declining queries by comparing two consecutive periods."
    (recent_start, recent_end), (prev_start, prev_end) = get_date_ranges_for_comparison(days)
    return compare_periods(
        get_top_queries(session, site_url, recent_start, recent_end, page_path=page_path, limit=limit),
        get_top_queries(session, site_url, prev_start, prev_end, page_path=page_path, limit=limit))
